<a href="https://colab.research.google.com/github/Sopitas24/CPSN/blob/main/Pandas_para_Ciencia_de_Datos_II_Proyecto_I_%E2%80%93_Parte_III_(Core)_javiera_cortes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

06 Pandas para Ciencia de Datos II - Proyecto I – Parte III (Core).txt

import io
import pandas as pd
from google.colab import files
from IPython.display import display

print("Selecciona el archivo CSV utilizado en la Parte II.")
archivos = files.upload()

if len(archivos) != 1:
    raise ValueError("Selecciona un solo archivo CSV.")

nombre_archivo = list(archivos.keys())[0]

if not nombre_archivo.lower().endswith(".csv"):
    raise ValueError("El archivo debe tener extensión .csv.")

datos = pd.read_csv(io.BytesIO(archivos[nombre_archivo]))
datos.columns = datos.columns.str.strip()

columnas_necesarias = ["Producto", "Tienda", "Ventas", "Precio"]

faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in datos.columns
]

if faltantes:
    raise ValueError(
        f"Faltan estas columnas: {', '.join(faltantes)}. "
        "Utiliza el archivo correspondiente a la Parte II."
    )

print("\nPRIMERAS 10 FILAS")
display(datos.head(10))

for columna in ["Producto", "Tienda"]:
    datos[columna] = datos[columna].astype("string").str.strip()
    datos[columna] = datos[columna].replace("", pd.NA)

for columna in ["Ventas", "Precio"]:
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")
    datos[columna] = datos[columna].replace(
        [float("inf"), float("-inf")],
        float("nan")
    )

print("\nVALORES FALTANTES EN LAS COLUMNAS NECESARIAS")
print(datos[columnas_necesarias].isna().sum())

filas_antes = len(datos)
datos = datos.dropna(subset=columnas_necesarias).copy()

print("Filas eliminadas por datos faltantes:", filas_antes - len(datos))

filas_validas = (
    (datos["Ventas"] >= 0)
    & (datos["Ventas"] % 1 == 0)
    & (datos["Precio"] >= 0)
)

print("Filas con ventas o precios inválidos:", (~filas_validas).sum())
datos = datos.loc[filas_validas].copy()

if datos.empty:
    raise ValueError("No quedaron datos válidos para analizar.")

datos["Ventas"] = datos["Ventas"].astype(int)
datos = datos.reset_index(drop=True)

datos["Ingreso_Total"] = datos["Ventas"] * datos["Precio"]

venta_minima = datos["Ventas"].min()
venta_maxima = datos["Ventas"].max()

if venta_maxima > venta_minima:
    datos["Ventas_Normalizadas"] = (
        (datos["Ventas"] - venta_minima)
        / (venta_maxima - venta_minima)
    )
else:

    datos["Ventas_Normalizadas"] = 0.0

datos["Nivel_Venta"] = pd.cut(
    datos["Ventas"],
    bins=[-float("inf"), 30, 50, float("inf")],
    labels=["Baja", "Media", "Alta"]
)

if "Fecha" in datos.columns:
    datos["Fecha"] = pd.to_datetime(
        datos["Fecha"],
        errors="coerce"
    )

    datos["Mes"] = datos["Fecha"].dt.to_period("M")

    print("\nFechas faltantes o inválidas:", datos["Fecha"].isna().sum())
    print("Esas filas se conservarán, pero no entrarán en el resumen mensual.")

print("\nDATOS CON LAS NUEVAS COLUMNAS")
display(datos.head(10))

resumen_producto_tienda = datos.groupby(
    ["Producto", "Tienda"]
).agg(
    Ventas_Totales=("Ventas", "sum"),
    Venta_Promedio=("Ventas", "mean"),
    Cantidad_Registros=("Ventas", "count"),
    Venta_Minima=("Ventas", "min"),
    Venta_Maxima=("Ventas", "max"),
    Desviacion_Estandar=("Ventas", "std"),
    Varianza=("Ventas", "var"),
    Ingresos_Totales=("Ingreso_Total", "sum")
).reset_index()

resumen_producto_tienda = resumen_producto_tienda.sort_values(
    "Ingresos_Totales",
    ascending=False
)

print("\nRESUMEN POR PRODUCTO Y TIENDA")
display(resumen_producto_tienda.round(2))

if "Mes" in datos.columns:
    resumen_producto_mes = datos.dropna(subset=["Mes"]).groupby(
        ["Producto", "Mes"]
    ).agg(
        Ventas_Totales=("Ventas", "sum"),
        Venta_Promedio=("Ventas", "mean"),
        Cantidad_Registros=("Ventas", "count"),
        Venta_Minima=("Ventas", "min"),
        Venta_Maxima=("Ventas", "max"),
        Desviacion_Estandar=("Ventas", "std"),
        Varianza=("Ventas", "var"),
        Ingresos_Totales=("Ingreso_Total", "sum")
    ).reset_index()

    print("\nRESUMEN POR PRODUCTO Y MES")
    display(resumen_producto_mes.round(2))

else:
    print("\nNo se creó el resumen mensual porque el archivo no tiene Fecha.")

datos["Media_Grupo"] = datos.groupby(
    ["Producto", "Tienda"]
)["Ventas"].transform("mean")


def analizar_venta(fila):

    desviacion = fila["Ventas"] - fila["Media_Grupo"]

    if fila["Media_Grupo"] != 0:
        porcentaje = desviacion / fila["Media_Grupo"] * 100
    else:
        porcentaje = float("nan")


    if abs(desviacion) < 0.000001:
        comparacion = "Igual a la media"
    elif desviacion > 0:
        comparacion = "Sobre la media"
    else:
        comparacion = "Bajo la media"

    return pd.Series({
        "Desviacion_Unidades": desviacion,
        "Desviacion_Porcentaje": porcentaje,
        "Comparacion_Grupo": comparacion
    })

resultados_personalizados = datos.apply(analizar_venta, axis=1)

datos[
    [
        "Desviacion_Unidades",
        "Desviacion_Porcentaje",
        "Comparacion_Grupo"
    ]
] = resultados_personalizados

print("\nVENTAS COMPARADAS CON LA MEDIA DE SU GRUPO")
display(
    datos[
        [
            "Producto",
            "Tienda",
            "Ventas",
            "Media_Grupo",
            "Desviacion_Unidades",
            "Desviacion_Porcentaje",
            "Comparacion_Grupo"
        ]
    ].head(10).round(2)
)

print("\nRESUMEN POR NIVEL DE VENTA")

resumen_nivel = datos.groupby(
    "Nivel_Venta",
    observed=True
).agg(
    Cantidad_Registros=("Ventas", "count"),
    Ventas_Totales=("Ventas", "sum"),
    Ingresos_Totales=("Ingreso_Total", "sum")
)

display(resumen_nivel.round(2))

print("\nCOMPARACIÓN RESPECTO A LA MEDIA DEL GRUPO")
print(datos["Comparacion_Grupo"].value_counts())

mayor_ingreso = resumen_producto_tienda["Ingresos_Totales"].max()

mejores_grupos = resumen_producto_tienda[
    resumen_producto_tienda["Ingresos_Totales"] == mayor_ingreso
]

print("\nCOMBINACIONES DE PRODUCTO Y TIENDA CON MAYOR INGRESO")
display(
    mejores_grupos[
        ["Producto", "Tienda", "Ventas_Totales", "Ingresos_Totales"]
    ].round(2)
)

print("\nCONCLUSIONES GENERALES")
print("Registros analizados:", len(datos))
print("Productos diferentes:", datos["Producto"].nunique())
print("Tiendas diferentes:", datos["Tienda"].nunique())
print(f"Unidades vendidas: {datos['Ventas'].sum():,}")
print(f"Ingresos totales: {datos['Ingreso_Total'].sum():,.2f}")
print("Los ingresos no representan ganancias: no contamos con los costos.")

datos.to_csv(
    "ventas_transformadas.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_producto_tienda.to_csv(
    "resumen_producto_tienda.csv",
    index=False,
    encoding="utf-8-sig"
)

if "Mes" in datos.columns:
    resumen_producto_mes.to_csv(
        "resumen_producto_mes.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("\nAnálisis finalizado.")
print("Los archivos generados están en el panel de archivos de Colab.")

Mostrando 06 Pandas para Ciencia de Datos II - Proyecto I – Parte III (Core).txt